# 02 · Bin placement model (MCLP)

**Owner: Ronnie.** Only Ronnie edits this notebook (one owner per file).

This notebook picks where to put bin stations so they cover as much waste-generating foot traffic as possible.
It reads the files described in [`docs/data-contract.md`](../docs/data-contract.md) and writes `outputs/chosen_sites.csv`.

**Honesty rule:** results are projected **coverage**, never measured diversion. Values marked `PLACEHOLDER` are assumptions from `binopt/config.py`.

Start with the **example data** (8 made-up points, answers known). Switch to the real campus data once your model gets the example right.

## 0. Setup

Run this cell first. In Google Colab it downloads the repo and installs the libraries (about 1 minute). On a laptop it just makes sure we're in the repo folder.

In [ ]:
# Simple: get the project files and libraries ready.
# Technical: in Colab, clone the repo into /content and pip install requirements;
# add the repo folder to sys.path so `import binopt` works without restarting.
import os, sys

REPO_URL = "https://github.com/Rondo12/CoverageBinCA---CBIO-Collaborative-Innovation-Design-Challenge.git"
REPO_DIR = "/content/CoverageBinCA---CBIO-Collaborative-Innovation-Design-Challenge"

if "google.colab" in sys.modules:           # True only when running in Colab
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                          # on a laptop: move from notebooks/ up to the repo folder

sys.path.insert(0, os.getcwd())
print("Working in:", os.getcwd())

## 1. Load the data

`DATA_DIR = "data/example"` uses the tiny example. Change it to `"data"` for the real campus.

Settings come from `binopt/config.py` (Eric's file): change them there, not here.

In [ ]:
import pandas as pd
from binopt.config import COVERAGE_RADII_M, DEFAULT_RADIUS_M, N_SITES

DATA_DIR = "data/example"   # switch to "data" for the real campus

demand = pd.read_csv(f"{DATA_DIR}/demand.csv")          # where waste is generated (weight = estimated demand)
candidates = pd.read_csv(f"{DATA_DIR}/candidates.csv")  # where a bin station could go
distances = pd.read_csv(f"{DATA_DIR}/distances.csv")    # demand_id, site_id, distance_m (straight line)

if DATA_DIR == "data/example":
    N_SITES = 2   # the example's known answers are for p = 2

RADIUS_M = DEFAULT_RADIUS_M   # PLACEHOLDER
print(f"{len(demand)} demand points, {len(candidates)} candidate sites, {len(distances)} distance rows")
print(f"Choosing {N_SITES} sites, coverage radius {RADIUS_M} m")
demand.head()

## 2. The model in plain words

**Maximal Covering Location Problem (MCLP).**

> You have **p** bin stations. Put them at candidate sites so that the total **weight** of demand points with a station within **R** meters is as large as possible.

The pieces:

| piece | plain words | math |
|---|---|---|
| decision: build? | for each site, yes/no: do we put a station there? | $x_j \in \{0,1\}$ |
| decision: covered? | for each demand point, yes/no: is it covered? | $y_i \in \{0,1\}$ |
| objective | make the covered weight as big as possible | maximize $\sum_i w_i \, y_i$ |
| budget | exactly p stations | $\sum_j x_j = p$ |
| coverage rule | a point can only count as covered if some chosen site is within R | $y_i \le \sum_{j \in N_i} x_j$ |

$N_i$ = the sites within R meters of demand point *i* (rows of `distances.csv` with `distance_m <= R`).

Fill in the cells below. Each one has hints in comments; the code is yours to write.

## 3. Coverage sets: which sites are within R of each demand point?

In [ ]:
# TODO (Ronnie): build N_i for every demand point.
#
# Hints:
#   - Filter `distances` to rows where distance_m <= RADIUS_M.
#     (pandas: df[df["col"] <= value] keeps only the rows where the condition is True)
#   - Then group by demand_id to get a list of site_ids for each demand point.
#     (pandas: df.groupby("a")["b"].apply(list) -> {a: [b, b, ...]})
#   - A demand point with no nearby sites simply can't be covered. That's OK.
#
# Check on the example at 50 m: D1 -> [S1], D2 -> [S1], D8 -> [] (nothing within 50 m)


## 4. Create the problem and the decision variables

In [ ]:
# TODO (Ronnie): create the PuLP problem and the variables x_j and y_i.
#
# Hints:
#   import pulp
#   - pulp.LpProblem("name", pulp.LpMaximize) makes an empty maximization problem.
#   - pulp.LpVariable("name", cat="Binary") makes one yes/no (0 or 1) variable.
#   - pulp.LpVariable.dicts("x", list_of_ids, cat="Binary") makes one per id, as a dictionary.
#     Simple: a dictionary is a lookup table, x["S1"] gives the variable for site S1.


## 5. Objective: maximize covered weight

In [ ]:
# TODO (Ronnie): add the objective  sum over i of  w_i * y_i
#
# Hints:
#   - problem += <expression>   (the first expression you add without a comparison is the objective)
#   - pulp.lpSum(...) adds up many terms, e.g. pulp.lpSum(w[i] * y[i] for i in ids)
#   - Get the weights as a lookup: w = demand.set_index("demand_id")["weight"]


## 6. Constraints

In [ ]:
# TODO (Ronnie): add the two rules
#   (a) budget:   sum of x_j == N_SITES
#   (b) coverage: y_i <= sum of x_j over sites j in N_i   (one rule per demand point)
#
# Hints:
#   - problem += (expression == value), "a_name"      adds a constraint with a name
#   - For a demand point with no nearby sites, the rule becomes y_i <= 0 (it can't be covered).


## 7. Solve

In [ ]:
# TODO (Ronnie): solve and check it worked
#
# Hints:
#   - problem.solve(pulp.PULP_CBC_CMD(msg=False))   # CBC is the free solver that ships with PuLP
#   - pulp.LpStatus[problem.status] should be "Optimal"
#   - x[j].value() is 1 for chosen sites, 0 otherwise


## 8. Build the output table (data contract format)

Required columns: `site_id, lat, lon, covered_weight, covered_share, radius_m, n_sites`.

**No double counting:** credit each covered demand point only to its **nearest** chosen site. Then the per-site numbers add up to the total.

In [ ]:
# TODO (Ronnie): make a DataFrame called `chosen` with the columns above, one row per chosen site.
#
# Hints:
#   - chosen_ids = [j for j in candidates["site_id"] if x[j].value() > 0.5]
#   - For "nearest chosen site": filter distances to chosen sites and distance_m <= RADIUS_M,
#     sort by distance_m, then drop_duplicates("demand_id") keeps each point's nearest site.
#   - covered_share = covered_weight / demand["weight"].sum()
#   - Chosen sites that end up covering nothing still get a row, with covered_weight 0.

chosen = None  # replace with your table


## 9. Check against the known example answer

Only meaningful while `DATA_DIR = "data/example"`.

In [ ]:
# Compares your `chosen` table with the hand-worked answer for this radius.
if chosen is None:
    print("Build `chosen` in step 8 first.")
elif DATA_DIR != "data/example":
    print("Skipped: this check only applies to the example data.")
else:
    expected = pd.read_csv("data/example/expected_chosen_sites.csv")
    expected = expected[expected["radius_m"] == RADIUS_M]
    got = dict(zip(chosen["site_id"], chosen["covered_weight"]))
    want = dict(zip(expected["site_id"], expected["covered_weight"]))
    if got == want:
        print(f"Correct for {RADIUS_M} m: {want}")
    else:
        print(f"Not yet. Expected {want}, got {got}")

## 10. Report and save

Say it as **projected coverage**, never diversion.

In [ ]:
import os
if chosen is not None:
    total = chosen["covered_share"].sum()
    print(f"Projected coverage: {total:.1%} of estimated demand within {RADIUS_M} m of a station "
          f"({len(chosen)} stations). Estimates use PLACEHOLDER weights.")
    os.makedirs("outputs", exist_ok=True)
    chosen.to_csv("outputs/chosen_sites.csv", index=False)
    print("Saved outputs/chosen_sites.csv")
    # In Colab, "Save a copy in GitHub" saves only this notebook, not the CSV.
    # This downloads the CSV so you can upload it on github.com (outputs/ > Add file > Upload files).
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download("outputs/chosen_sites.csv")

## 11. (Optional) Sensitivity check: 3 radii

Run the model once per radius in `COVERAGE_RADII_M` (PLACEHOLDER: 30 / 50 / 75 m) and stack the results in one `chosen_sites.csv`; the `radius_m` column tells the runs apart.
Tip: once steps 3–8 work, move them into a function `solve(radius_m)` that returns `chosen`, then:
```python
chosen = pd.concat([solve(r) for r in COVERAGE_RADII_M])
```

In [ ]:
# TODO (Ronnie, optional): sensitivity check
